# Sobrevivencia de negocios: los 32 estados

Junta lo que dejó `por_estado.py` (`resultados/NN.json`, una corrida de `estado.ipynb` por estado). Mismo diseño en todos: DENUE 2019_11 contra 2024_11, el estado completo, efecto fijo por giro y errores por bloques de 2 km. Ninguna probabilidad pasa a Mapo (RNF5).

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd

from sobrevivencia import datos, graficas

r = datos.resultados(Path("resultados"))
f"{len(r)} estados, {r['negocios'].sum():,} negocios" 

'32 estados, 5,447,246 negocios'

## $\color{#dda}{\text{1. Cortes censales}}$

Parte de los negocios que sigue de 2018_11 a 2019_11 y de 2023_11 a 2024_11 (cortes censales) contra la de 2019_11 a 2023_11 (cuatro años sin censo).

In [2]:
r[["nombre", "negocios", "corte_2019", "entre_cortes", "corte_2024", "sigue"]].sort_values("sigue").reset_index(drop=True)

,nombre,negocios,corte_2019,entre_cortes,corte_2024,sigue
0,Quintana Roo,64714,53.8,92.8,65.0,60.17
1,Morelos,109806,58.5,95.0,63.3,60.39
2,Colima,39213,57.7,87.5,68.4,60.63
3,Baja California Sur,35924,58.1,89.6,68.6,61.49
4,Tabasco,83978,59.9,95.0,65.0,61.59
5,Campeche,41418,58.2,92.3,68.4,63.32
6,Nuevo León,179233,60.6,94.7,67.2,63.64
7,Querétaro,92395,58.7,94.1,68.0,63.82
8,Tlaxcala,81102,60.0,94.8,67.5,64.14
9,Nayarit,65887,61.9,93.6,68.6,64.42


## $\color{#dda}{\text{2. Vecinos del mismo giro}}$

In [3]:
graficas.bosque(r, "vecinos_mismo", "Vecinos del mismo giro a 200 m (log), dentro del giro").show()

In [4]:
claros = r[(r["vecinos_mismo_desde"] > 0) | (r["vecinos_mismo_hasta"] < 0)]
print(f"Positivos y claros: {(claros['vecinos_mismo_puntos'] > 0).sum()}; negativos y claros: {(claros['vecinos_mismo_puntos'] < 0).sum()}; cruzan 0: {len(r) - len(claros)}")
print(f"Promedio ponderado por negocios: {(r['vecinos_mismo_puntos'] * r['negocios']).sum() / r['negocios'].sum():.2f} puntos")
r[["nombre", "vecinos_mismo_puntos", "vecinos_mismo_desde", "vecinos_mismo_hasta", "vecinos_mismo_rereg_puntos"]].sort_values("vecinos_mismo_puntos").round(2).reset_index(drop=True)

Positivos y claros: 20; negativos y claros: 0; cruzan 0: 12
Promedio ponderado por negocios: 1.14 puntos


,nombre,vecinos_mismo_puntos,vecinos_mismo_desde,vecinos_mismo_hasta,vecinos_mismo_rereg_puntos
0,Nayarit,-0.10,-1.49,1.30,-0.07
1,Tabasco,0.04,-0.73,0.81,0.20
2,Ciudad de México,0.11,-0.72,0.94,0.55
3,Baja California Sur,0.26,-1.04,1.56,0.61
4,Zacatecas,0.34,-0.37,1.06,0.65
5,Sonora,0.43,-0.23,1.09,0.37
6,Sinaloa,0.52,-0.14,1.19,0.57
7,Baja California,0.62,-0.63,1.86,0.74
8,Morelos,0.71,-0.48,1.90,1.04
9,Durango,0.78,-0.18,1.74,0.85


## $\color{#dda}{\text{3. Giros que se agrupan}}$

Interacción: cuánto más suman los vecinos del giro en los giros que se agrupan en ese estado (distancia al vecino del giro menos del 80 % de la del azar).

In [5]:
graficas.bosque(r, "interaccion", "Vecinos del giro × giro que se agrupa").show()

## $\color{#dda}{\text{4. Cualquier giro y plazas}}$

In [6]:
graficas.bosque(r, "vecinos_todos", "Vecinos de cualquier giro a 200 m (log)").show()

In [7]:
graficas.bosque(r, "en_plaza", "En plaza, mercado o edificio comercial").show()

## $\color{#dda}{\text{5. Qué tanto se puede predecir}}$

In [8]:
r[["nombre", "auc_giro", "auc_arboles"]].assign(ganancia=lambda d: d["auc_arboles"] - d["auc_giro"]).describe().round(3)

,auc_giro,auc_arboles,ganancia
count,32.000,32.000,32.000
mean,0.617,0.682,0.065
std,0.012,0.012,0.008
min,0.597,0.657,0.045
25%,0.607,0.673,0.060
50%,0.618,0.686,0.066
75%,0.625,0.690,0.070
max,0.637,0.702,0.080
